Step 1 — Connect to a real API

We'll use the GitHub REST API because it gives us clean structured JSON and doesn't require an API key for basic public repository data.

In [4]:
import requests
#requests is a Python library used to make HTTP requests.

url = "https://api.github.com/repos/langchain-ai/langchain"

response = requests.get(url)

print("Status code:", response.status_code)
print("Response type:", type(response.json()))

Status code: 200
Response type: <class 'dict'>


Step 2 — Inspect the actual API data

In [7]:
data = response.json()
# Actual API data converted into a Python dictionary

print(data.keys())

dict_keys(['id', 'node_id', 'name', 'full_name', 'private', 'owner', 'html_url', 'description', 'fork', 'url', 'forks_url', 'keys_url', 'collaborators_url', 'teams_url', 'hooks_url', 'issue_events_url', 'events_url', 'assignees_url', 'branches_url', 'tags_url', 'blobs_url', 'git_tags_url', 'git_refs_url', 'trees_url', 'statuses_url', 'languages_url', 'stargazers_url', 'contributors_url', 'subscribers_url', 'subscription_url', 'commits_url', 'git_commits_url', 'comments_url', 'issue_comment_url', 'contents_url', 'compare_url', 'merges_url', 'archive_url', 'downloads_url', 'issues_url', 'pulls_url', 'milestones_url', 'notifications_url', 'labels_url', 'releases_url', 'deployments_url', 'created_at', 'updated_at', 'pushed_at', 'git_url', 'ssh_url', 'clone_url', 'svn_url', 'homepage', 'size', 'stargazers_count', 'watchers_count', 'language', 'has_issues', 'has_projects', 'has_downloads', 'has_wiki', 'has_pages', 'has_discussions', 'forks_count', 'mirror_url', 'archived', 'disabled', 'open_

In [6]:
print("Repository:", data["full_name"])
print("Description:", data["description"])
print("Stars:", data["stargazers_count"])
print("Forks:", data["forks_count"])
print("Language:", data["language"])

Repository: langchain-ai/langchain
Description: The agent engineering platform.
Stars: 147335
Forks: 24669
Language: Python


AI application that uses external tools/APIs first needs to reliably obtain and structure the external data.


requests.get()
      ↓
Response object
      ↓
response.json()
      ↓
Python dictionary
      ↓
data["field"]

Now let's deliberately inspect the raw JSON structure before touching the LLM.

In [8]:
import json

print(json.dumps(data, indent=2)[:3000])

{
  "id": 552661142,
  "node_id": "R_kgDOIPDwlg",
  "name": "langchain",
  "full_name": "langchain-ai/langchain",
  "private": false,
  "owner": {
    "login": "langchain-ai",
    "id": 126733545,
    "node_id": "O_kgDOB43M6Q",
    "avatar_url": "https://avatars.githubusercontent.com/u/126733545?v=4",
    "gravatar_id": "",
    "url": "https://api.github.com/users/langchain-ai",
    "html_url": "https://github.com/langchain-ai",
    "followers_url": "https://api.github.com/users/langchain-ai/followers",
    "following_url": "https://api.github.com/users/langchain-ai/following{/other_user}",
    "gists_url": "https://api.github.com/users/langchain-ai/gists{/gist_id}",
    "starred_url": "https://api.github.com/users/langchain-ai/starred{/owner}{/repo}",
    "subscriptions_url": "https://api.github.com/users/langchain-ai/subscriptions",
    "organizations_url": "https://api.github.com/users/langchain-ai/orgs",
    "repos_url": "https://api.github.com/users/langchain-ai/repos",
    "event

STEP 3 : We're eventually going to transform messy API responses into clean information for the LLM.

We might construct :

research_data = {
    "repository": data["full_name"],
    "description": data["description"],
    "stars": data["stargazers_count"],
    "forks": data["forks_count"],
    "language": data["language"],
    "owner": data["owner"]["login"]
}

In [9]:
research_data = {
    "repository": data["full_name"],
    "description": data["description"],
    "stars": data["stargazers_count"],
    "forks": data["forks_count"],
    "language": data["language"],
    "owner": data["owner"]["login"]
}

print(research_data)

{'repository': 'langchain-ai/langchain', 'description': 'The agent engineering platform.', 'stars': 147335, 'forks': 24669, 'language': 'Python', 'owner': 'langchain-ai'}


In [14]:
import json

print(json.dumps(research_data, indent=2))

{
  "repository": "langchain-ai/langchain",
  "description": "The agent engineering platform.",
  "stars": 147335,
  "forks": 24669,
  "language": "Python",
  "owner": "langchain-ai"
}


STEP 4 : Connect this exact object → Ollama, and that's where the AI part starts.

In [15]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0
)

Step 5 — Give the API data to the LLM

In [16]:
prompt = f"""
You are a research assistant.

Analyze the following GitHub repository information:

Repository: {research_data["repository"]}
Description: {research_data["description"]}
Stars: {research_data["stars"]}
Forks: {research_data["forks"]}
Language: {research_data["language"]}
Owner: {research_data["owner"]}

Provide a concise research summary covering:
1. What the repository is
2. What technology it primarily uses
3. Its popularity based on the provided metrics

Use only the information provided above.
"""

response = llm.invoke(prompt)

print(response.content)

Here is a concise research summary:

**Repository Overview**

The langchain-ai/langchain repository is an open-source agent engineering platform.

**Technology and Programming Language**

The primary technology used in this repository is Python, indicating that it leverages Python's strengths for building and deploying AI models.

**Popularity Metrics**

Based on the provided metrics:
- The repository has 147,335 stars, suggesting a high level of interest and adoption among developers.
- It has 24,669 forks, implying a significant community-driven effort to extend or modify the platform.
- With over 147,000 stars and 24,600 forks, this repository demonstrates substantial popularity in the developer community.


Taking user input and grounding the response to the context

In [18]:
user_question = input("What would you like to know about this repository? ")

prompt = f"""
You are a research assistant.

Answer the user's question using ONLY the repository
information provided below.

Repository information:
{research_data}

User question:
{user_question}

If the provided information does not contain the answer,
say: "The available repository data does not contain this information."

Do not invent facts or make assumptions.
"""

response = llm.invoke(prompt)

print("\nResearch Answer:")
print(response.content)


Research Answer:
The available repository data does not contain this information.


From here we work in app.py because .ipynb is for experiments and .py is for real time 

1. Take in repository name from the user

2. Error handling ,e.g. what happens when the user enters abc/xyz or GitHub is unavailable.

   Be mindful this needs to be handled while fetching data so handle error in data fucntion directly

3. Validate API data before sending it to the LLM 

    Like suppose the required field is not exposed in the API, look and handle that; Make the extraction defensive